In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException, StaleElementReferenceException
from dotenv import dotenv_values
import time
import os

In [2]:
COURSE_NAME = "24CSE0212-Data Structures using Object Oriented Programming-II-2024-BE-CSE-4th Sem"
# COURSE_NAME = "24CSE0213-Linux Administration-2024-BE-CSE-4th Sem"
BROWSER = "CHROME"
HEADLESS_MODE = False

# Attempt questions even if they are complete (For development/testing purpose only)
ATTEMPT_EVEN_IF_COMPLETE = False

QUESTIONS_TO_SKIP = ["Form a new number", "FInd the numbers of shifts in Insertion sort"]

import re

def sanitize_filename(filename):
    """Must match exactly what download_answers.ipynb uses to create folder names."""
    if ":" in filename:
        parts = filename.split(":")
        filename = " _ ".join(part.strip() for part in parts if part.strip())
    invalid_chars = r'<>"/\|?*'
    for char in invalid_chars:
        filename = filename.replace(char, "")
    filename = filename.replace("&", "and")
    filename = filename.replace("...", "")
    return filename.strip()[:100]

def find_folder_by_prefix(parent_folder: str, prefix: str) -> str:
    """Finds a subfolder whose name starts with the given prefix.
    Tries the sanitized prefix first, then falls back to raw prefix."""
    if not os.path.isdir(parent_folder):
        return None
    sanitized_prefix = sanitize_filename(prefix)
    for name in os.listdir(parent_folder):
        full = os.path.join(parent_folder, name)
        if os.path.isdir(full) and name.startswith(sanitized_prefix):
            return name
    # fallback: try raw prefix (for folders that were never sanitized)
    for name in os.listdir(parent_folder):
        full = os.path.join(parent_folder, name)
        if os.path.isdir(full) and name.startswith(prefix):
            return name
    return None

SUPPORTED_EXTENSIONS = {".c", ".cpp", ".java", ".py", ".sql", ".sh"}

EXTENSION_MAPPING = {
    ".c": "C",
    ".cpp": "C++",
    ".java": "Java",
    ".py": "Python",
    ".sql": "SQL",
    ".sh": "Bash",
}

LANGUAGE_SYNONYMS = {
    "C": ["C", "C Language"],
    "C++": ["C++", "CPP"],
    "Java": ["Java"],
    "Python": ["Python", "Python 3"],
    "SQL": ["SQL"],
    "Bash": ["Bash", "Shell"],
}

def find_code_file(folder: str) -> str:
    """Returns the first code file with a supported extension found in folder."""
    if not os.path.isdir(folder):
        return None
    for fname in os.listdir(folder):
        ext = os.path.splitext(fname)[1].lower()
        if ext in SUPPORTED_EXTENSIONS:
            return os.path.join(folder, fname)
    return None


In [3]:
from selenium import webdriver

BROWSER_OPTIONS_MAP = {
    "CHROME": webdriver.ChromeOptions,
    "FIREFOX": webdriver.FirefoxOptions,
}
BROWSER_DRIVER_MAP = {
    "CHROME": webdriver.Chrome,
    "FIREFOX": webdriver.Firefox,
}

def init_driver(browser: str = "CHROME", headless: bool = False):
    """Initialize Selenium driver with options."""
    options = BROWSER_OPTIONS_MAP[browser]()
    
    if headless:
        options.add_argument("--headless=new")
    else:
        # Keep browser window open after script finishes (Chrome only)
        if browser == "CHROME":
            options.add_experimental_option("detach", True)

    driver = BROWSER_DRIVER_MAP[browser](options=options)
    driver.delete_all_cookies()
    wait = WebDriverWait(driver, 15)
    
    return driver, wait


In [4]:
def login_into_testpad():
    """Logs into Testpad using credentials from .env file."""
    global driver, wait
    
    try:
        # Create browser options here (no dependency on Cell 3)
        options = BROWSER_OPTIONS_MAP[BROWSER]()
        if HEADLESS_MODE:
            options.add_argument("-headless")
        else:
            options.add_argument("-detach")

        # Setup driver + wait
        driver = BROWSER_DRIVER_MAP[BROWSER](options=options)
        wait = WebDriverWait(driver, 15)

        TESTPAD_URL = "https://course.testpad.chitkara.edu.in"
        driver.get(TESTPAD_URL)

        # Switch to login iframe (if available)
        try:
            wait.until(EC.frame_to_be_available_and_switch_to_it((By.ID, "loginIframe")))
        except Exception:
            print("⚠️ No login iframe found, continuing in main frame")

        # Load credentials from .env
        creds = dotenv_values(".env")
        email = creds.get("TESTPAD_EMAIL")
        password = creds.get("TESTPAD_PASSWORD")
        if not email or not password:
            raise ValueError("Missing TESTPAD_EMAIL or TESTPAD_PASSWORD in .env file")

        # Fill form
        email_element = wait.until(EC.visibility_of_element_located((By.ID, "email")))
        email_element.clear()
        email_element.send_keys(email)

        password_element = driver.find_element(By.ID, "password")
        password_element.clear()
        password_element.send_keys(password)

        # Submit
        submit_button = wait.until(EC.element_to_be_clickable((By.ID, "submit")))
        driver.execute_script("arguments[0].click()", submit_button)

        # Exit iframe if inside one
        try:
            driver.switch_to.default_content()
        except Exception:
            pass

        print("✅ Login successful")

    except Exception as e:
        import traceback
        print("❌ Login failed:")
        traceback.print_exc()
        raise


In [5]:
def wait_for_loader_disappearance():
    try:
        wait.until(EC.invisibility_of_element_located((By.ID, "loader-container")))
    except TimeoutException:
        print("Loader did not disappear, continuing anyway...")

def click_course():
    # Wait until all modules are visible
    wait.until(EC.visibility_of_all_elements_located((By.CLASS_NAME, "learning-module-card")))

    # Re-find course card each time to avoid stale element
    try:
        learning_module_card = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, f"//a[text()='{COURSE_NAME}']/../.."))
        )

        learning_module_href = learning_module_card.find_element(
            By.CSS_SELECTOR,
            "[class='icons-link-container tooltip-up'][data-customtooltip='Learning Content']"
        )

        wait_for_loader_disappearance()
        WebDriverWait(driver, 10).until(EC.element_to_be_clickable(learning_module_href))
        driver.execute_script("arguments[0].click()", learning_module_href)

    except TimeoutException:
        raise ValueError(f"Course '{COURSE_NAME}' not found or not clickable.")

    # Wait for subtopics to load
    try:
        subtopics = wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "course-topics")))
        no_of_topics = len(subtopics)
    except TimeoutException:
        no_of_topics = 0

    if no_of_topics == 0:
        raise ValueError("No course topic showing!")

    print(f"Number of subtopics - {no_of_topics}")

    wait_for_loader_disappearance()

    global original_window
    original_window = driver.current_window_handle


In [6]:
def complete_coding_question(question_file_path):
    # Read code file
    with open(question_file_path, encoding="utf-8") as f:
        coding_question_file_content = f.read()

    print(f"📝 Code file loaded: {len(coding_question_file_content)} characters")

    # Determine required language from file extension
    codefile_extension = question_file_path[question_file_path.rindex("."):]
    required_language = EXTENSION_MAPPING.get(codefile_extension)
    if not required_language:
        raise ValueError(f"Unsupported file extension: {codefile_extension}")
    
    print(f"📋 Required language: {required_language}")

    # Get current language from dropdown - WITH BETTER ERROR HANDLING
    try:
        dropdown_language_selected = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "button[data-id='progLang']"))
        )
        current_language = dropdown_language_selected.text.strip()
        print(f"🌐 Current language dropdown: {current_language}")
    except TimeoutException as e:
        print(f"❌ Could not find language dropdown: {e}")
        print(f"Page HTML (first 1000 chars): {driver.page_source[:1000]}")
        raise ValueError("Language dropdown not found - check if question page loaded correctly")

    # Get all valid synonyms for required language
    required_language_options = LANGUAGE_SYNONYMS.get(required_language, [required_language])

    # Only switch if current language is not in the allowed options
    if current_language not in required_language_options:
        print(f"⚠️ Switching language from {current_language} to {required_language_options[0]}...")

        wait_for_loader_disappearance()
        driver.execute_script("arguments[0].click()", dropdown_language_selected)
        time.sleep(0.5)

        # Wait for dropdown to appear
        try:
            programming_languages_ul = WebDriverWait(driver, 3).until(
                EC.visibility_of_any_elements_located(
                    (By.CSS_SELECTOR, "ul[class='dropdown-menu inner show']"))
            )
            programming_languages_ul = [ul for ul in programming_languages_ul if ul.is_displayed()][0]
        except TimeoutException as e:
            print(f"❌ Dropdown menu did not appear: {e}")
            raise ValueError("Language dropdown menu failed to open")

        # Find and click the first matching language in synonyms
        programming_languages = programming_languages_ul.find_elements(By.TAG_NAME, "li")
        switched = False
        for language in programming_languages:
            try:
                language_text = language.find_element(By.CSS_SELECTOR, "a > span").text.strip()
                if language_text in required_language_options:
                    language.click()
                    switched = True
                    time.sleep(1)  # Wait for editor refresh
                    print(f"✅ Switched to {language_text}")
                    break
            except Exception as e:
                print(f"⚠️ Could not read language option: {e}")
                continue

        if not switched:
            raise ValueError(f"Could not switch to required language: {required_language_options[0]}")
    else:
        print(f"✅ Already at {current_language}, no switch needed.")

    # Inject code into CodeMirror editor - WITH BETTER ERROR HANDLING
    try:
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "CodeMirror")))
        print("✅ CodeMirror found, injecting code...")
        
        driver.execute_script(
            "const cm = document.getElementsByClassName('CodeMirror cm-s-default')[0].CodeMirror;"
            f"cm.setValue({coding_question_file_content!r});"
        )
        time.sleep(0.5)
        print("✅ Code injected successfully")
    except TimeoutException as e:
        print(f"❌ CodeMirror not found: {e}")
        raise ValueError("CodeMirror editor not found on page")
    except Exception as e:
        print(f"❌ Error injecting code: {e}")
        raise

    # Submit code - WITH BETTER ERROR HANDLING
    try:
        submit_code_button = driver.find_element(By.ID, "executeProgram")
        print("✅ Execute button found, clicking...")
        wait_for_loader_disappearance()
        driver.execute_script("arguments[0].click()", submit_code_button)
        print("✅ Execute button clicked")
    except Exception as e:
        print(f"❌ Could not find or click execute button: {e}")
        raise ValueError("Execute button not found")

    # Wait for test cases to appear - WITH RETRY
    test_cases = None
    for attempt in range(2):
        try:
            test_cases = WebDriverWait(driver, 5).until(
                EC.visibility_of_all_elements_located((By.CLASS_NAME, "test-cases"))
            )
            print(f"✅ Test cases appeared ({len(test_cases)} test cases)")
            break
        except TimeoutException:
            if attempt == 0:
                print("⚠️ First attempt to get test cases failed, retrying...")
                time.sleep(1)
                driver.execute_script("arguments[0].click()", submit_code_button)
            else:
                print("❌ Test cases did not appear after 2 attempts")
                raise TimeoutException("Test cases did not appear")

    # Check if all test cases passed
    try:
        test_cases_passed = driver.find_elements(
            By.CSS_SELECTOR, ".test-cases > img[src='/icons/check1.svg']"
        )
        print(f"✅ Test cases passed: {len(test_cases_passed)}/{len(test_cases)}")
        
        if len(test_cases_passed) != len(test_cases):
            raise ValueError(f"❌ Not all test cases passed ({len(test_cases_passed)}/{len(test_cases)})")
        else:
            print("✅ All test cases passed!")
    except Exception as e:
        print(f"❌ Error checking test results: {e}")
        raise

In [7]:
def complete_textarea_question(question_file_path):
    mq_question_file_content = open(question_file_path).read()

    # Wait for all questions
    questions = WebDriverWait(driver, 10).until(
        EC.visibility_of_all_elements_located(
            (By.CSS_SELECTOR, "#compileTestcases > [class='col-md-12 test_cases']")
        )
    )

    answers = mq_question_file_content.strip().split("\n\n\n")
    if len(questions) != len(answers):
        raise ValueError(f"{len(questions)} questions, {len(answers)} answers")

    # Fill all answers into corresponding textareas
    for i in range(len(questions)):
        driver.execute_script(
            f"const textarea = document.querySelector('#txtOutput{i}');"
            f"if(textarea) textarea.value = {answers[i]!r};"
        )

    time.sleep(0.3)
    wait_for_loader_disappearance()

    # Retry logic for Execute button
    for attempt in range(2):
        try:
            execute_program_button = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.ID, "executeProgram"))
            )
            driver.execute_script("arguments[0].click()", execute_program_button)
            
            outcomes = WebDriverWait(driver, 10).until(
                EC.visibility_of_all_elements_located(
                    (By.CSS_SELECTOR, "label.test_case_result > img.correct_result")
                )
            )
            break  # success
        except TimeoutException:
            if attempt == 0:
                print("Retrying Execute button click...")
            else:
                raise TimeoutException("Execute button failed twice.")

    # Count correct answers
    no_of_correct_answers = 0
    for outcome in outcomes:
        outcome_image_type = outcome.get_attribute("src")
        if outcome_image_type.endswith("/icons/check1.svg"):
            no_of_correct_answers += 1

    if no_of_correct_answers != len(questions):
        raise ValueError(
            f"Only {no_of_correct_answers}/{len(questions)} correct answers ❌"
        )
    else:
        print(f"✅ All {no_of_correct_answers} answers correct!")


In [8]:
def complete_mcq_question(question_file_path):
    mcq_question_file_content = open(question_file_path).read().strip()

    # ✅ Parse answer safely (handles "1" or "C) number")
    def parse_mcq_answer(raw_answer: str) -> int:
        raw_answer = raw_answer.strip()
        if raw_answer and raw_answer[0] in ['A', 'B', 'C', 'D']:
            return ord(raw_answer[0]) - ord('A') + 1
        return int(raw_answer)  # fallback for numeric answers

    mcq_answer = parse_mcq_answer(mcq_question_file_content)

    mcq_options = wait.until(EC.visibility_of_all_elements_located((
        By.CSS_SELECTOR, "[class='col-12 mt-2 div-radio-btn-choose-answer']")))

    wait_for_loader_disappearance()
    mcq_options[mcq_answer - 1].find_element(By.CLASS_NAME, "choose-answer-checkmark").click()

    submit_button = driver.find_element(By.CSS_SELECTOR, "input#executeMCQ")
    wait_for_loader_disappearance()
    submit_button.click()

    # ✅ Wait for result icon or "Next" button
    try:
        WebDriverWait(driver, 5).until(
            EC.any_of(
                EC.visibility_of_element_located((By.CSS_SELECTOR, "label.test_case_result > img")),
                EC.element_to_be_clickable((By.CSS_SELECTOR, "input#nextQuestion"))
            )
        )
    except TimeoutException:
        print("⚠️ No result icon or next button appeared, continuing anyway")

    # ✅ Click "Next" button if present
    try:
        next_button = driver.find_element(By.CSS_SELECTOR, "input#nextQuestion")
        if next_button.is_displayed() and next_button.is_enabled():
            driver.execute_script("arguments[0].click()", next_button)
    except:
        pass


In [9]:
def complete_tutorial_manual_run():
    """Pauses at tutorial, waits for user to click Next/Finish, then continues."""
    from selenium.common.exceptions import InvalidSessionIdException
    
    print("▶️ Tutorial opened. Please click 'Run' manually or wait for automation to continue.")
    
    max_wait_time = 120  # Wait maximum 2 minutes for user interaction
    start_time = time.time()
    check_interval = 2  # Check every 2 seconds

    while True:
        try:
            # Check if we've exceeded max wait time
            elapsed = time.time() - start_time
            if elapsed > max_wait_time:
                print(f"⚠️ Tutorial timeout after {max_wait_time}s, skipping...")
                break
            
            # Check if session is still valid
            try:
                driver.current_url  # Quick session check
            except InvalidSessionIdException:
                print("❌ Browser session lost, abandoning tutorial")
                raise
            
            # Check for Next button
            next_buttons = driver.find_elements(By.CSS_SELECTOR, "button.next-button")
            if next_buttons and any(btn.is_displayed() for btn in next_buttons):
                next_buttons[0].click()
                print("▶️ Next button clicked by automation")
                time.sleep(1)
                continue

            # Check for Finish button
            finish_buttons = driver.find_elements(By.CSS_SELECTOR, "button.finish-button")
            if finish_buttons and any(btn.is_displayed() for btn in finish_buttons):
                finish_buttons[0].click()
                print("✅ Tutorial completed by automation")
                break

            # Wait before checking again
            time.sleep(check_interval)

        except InvalidSessionIdException as ise:
            print(f"❌ Session lost during tutorial: {ise}")
            raise
        except Exception as e:
            print(f"⚠️ Error in tutorial: {e}, retrying...")
            time.sleep(check_interval)

In [10]:
import traceback, datetime, os

def _dump_debug(prefix="debug"):
    """Save screenshot and page HTML for inspection."""
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    png = f"{prefix}_{ts}.png"
    html = f"{prefix}_{ts}.html"
    try:
        driver.save_screenshot(png)
        print(f"Saved screenshot: {png}")
    except Exception as e:
        print("Failed to save screenshot:", e)
    try:
        with open(html, "w", encoding="utf-8") as f:
            f.write(driver.page_source)
        print(f"Saved page source: {html}")
    except Exception as e:
        print("Failed to save page source:", e)
    return png, html

def complete_course_subtopic_debug(subtopic_number: str, subtopic_name: str, subtopic_questions: list):
    """
    Debug variant of your function — robust logging + automatic recovery.
    Replace the original with this while debugging.
    """
    no_of_subtopic_questions = len(subtopic_questions)
    print(f"DEBUG: Subtopic {subtopic_number} - {subtopic_name} ({no_of_subtopic_questions} questions)")

    # iterate using index because the page may change
    for idx in range(no_of_subtopic_questions):
        try:
            question = subtopic_questions[idx]
            print("\n" + "="*60)
            print(f"Question index {idx+1}/{no_of_subtopic_questions}")

            # Basic info
            try:
                qnum = question.find_element(By.CLASS_NAME, "course-question-number").text
                qname = question.find_element(By.CLASS_NAME, "course-question-name").text
                qtype = question.find_element(By.CLASS_NAME, "course-question-type").text
            except Exception:
                qnum = qname = qtype = "<couldn't read>"
            print(f"Q: {qnum} | Name: {qname} | Type: {qtype}")
            print("Displayed:", question.is_displayed())
            print("Window handles BEFORE click:", driver.window_handles)
            print("Current window handle:", driver.current_window_handle)

            # Show a bit of the question HTML so we can verify selectors
            try:
                outer = question.get_attribute("outerHTML")
                print("Question outerHTML (first 500 chars):")
                print(outer[:500].replace("\n"," "))
            except Exception as e:
                print("Couldn't get outerHTML:", e)

            # Remember original window for robust switching
            original_window = driver.current_window_handle

            # Try to find and click the element that opens the question
            try:
                modal_btn = question.find_element(By.CLASS_NAME, "isQuestionModal")
                print("Found isQuestionModal, outerHTML preview:")
                print(modal_btn.get_attribute("outerHTML")[:300].replace("\n"," "))
                # scroll into view then click via JS
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", modal_btn)
                driver.execute_script("arguments[0].click()", modal_btn)
            except Exception as e:
                print("Error finding/clicking isQuestionModal:", e)
                _dump_debug(f"click_fail_q{idx+1}")
                raise

            # Wait for either new window or an in-page modal
            try:
                WebDriverWait(driver, 8).until(lambda d: len(d.window_handles) > 1)
                new_windows = [w for w in driver.window_handles if w != original_window]
                if new_windows:
                    new_window = new_windows[0]
                    driver.switch_to.window(new_window)
                    print("Switched to new window:", driver.current_window_handle)
                    print("New window URL:", driver.current_url)
                    _dump_debug(f"opened_q{idx+1}")
                    in_new_window = True
                else:
                    in_new_window = False
            except TimeoutException:
                # No new window — maybe an in-page modal opened
                print("No new window opened; checking for in-page modal...")
                in_new_window = False

            # If no new window, try to detect in-page modal container
            if not in_new_window:
                try:
                    modal_container = WebDriverWait(driver, 5).until(
                        EC.visibility_of_any_elements_located((By.CSS_SELECTOR, ".modal-dialog, .question-modal, .question-panel"))
                    )
                    print("Detected in-page modal. Proceeding in same window.")
                    _dump_debug(f"modal_q{idx+1}")
                except Exception:
                    print("No in-page modal detected.")
                    _dump_debug(f"no_window_or_modal_q{idx+1}")
                    raise RuntimeError("No new window and no modal detected after clicking question.")

            # --- Now attempt the question according to its type ---
            try:
                # re-evaluate qtype from the live DOM (in case of prior stale data)
                try:
                    qtype_live = driver.find_element(By.CLASS_NAME, "question-type").text
                except Exception:
                    qtype_live = qtype  # fallback to earlier read

                # determine path to solution file (reuse your existing logic)
                question_file_path = os.path.join("Testpad-Solutions", COURSE_NAME)
                # find subtopic folder
                for folder in os.listdir(question_file_path):
                    if folder.startswith(subtopic_number + ". "):
                        question_file_path = os.path.join(question_file_path, folder)
                        break
                # find question folder
                for folder in os.listdir(question_file_path):
                    if folder.startswith(qnum + ". "):
                        question_file_path = os.path.join(question_file_path, folder)
                        break
                if qtype.lower().startswith("coding"):
                    # find source file by extension
                    chosen_file = None
                    for f in os.listdir(question_file_path):
                        if any(f.endswith(ext) for ext in EXTENSION_MAPPING.keys()):
                            chosen_file = os.path.join(question_file_path, f)
                            break
                    if not chosen_file:
                        raise FileNotFoundError("No code file found in question folder.")
                    print("Using code file:", chosen_file)
                    complete_coding_question(chosen_file)
                elif qtype.lower().startswith("mcq"):
                    answer_file = os.path.join(question_file_path, "Answer.txt")
                    complete_mcq_question(answer_file)
                elif qtype.lower().startswith("mq") or qtype.lower().startswith("textarea"):
                    answer_file = os.path.join(question_file_path, "Answer.txt")
                    complete_textarea_question(answer_file)
                elif qtype.lower().startswith("tutorial"):
                    complete_tutorial_manual_run()
                else:
                    # Unknown type — try your original handlers
                    print("Unknown question type; trying handlers in fallback mode.")
                    try:
                        complete_coding_question(os.path.join(question_file_path, "Answer.txt"))
                    except Exception:
                        pass

                print("Question handler finished successfully.")

            except Exception as e:
                print("Exception while running handler:", e)
                traceback.print_exc()
                _dump_debug(f"handler_exception_q{idx+1}")
                # attempt one recovery refresh and retry once
                try:
                    print("Attempting recovery: refresh and retry once.")
                    driver.refresh()
                    wait_for_loader_disappearance()
                    time.sleep(1)
                    # Retry handler once (same logic, but simplified)
                    if qtype.lower().startswith("coding"):
                        complete_coding_question(chosen_file)
                    elif qtype.lower().startswith("mcq"):
                        complete_mcq_question(answer_file)
                    elif qtype.lower().startswith("mq") or qtype.lower().startswith("textarea"):
                        complete_textarea_question(answer_file)
                    print("Retry succeeded.")
                except Exception as e2:
                    print("Retry failed:", e2)
                    traceback.print_exc()
                    _dump_debug(f"retry_failed_q{idx+1}")

            # --- Close question view and return to original window/modal state ---
            try:
                if in_new_window:
                    # close new window, switch back
                    driver.close()
                    # switch back to original if still present
                    if original_window in driver.window_handles:
                        driver.switch_to.window(original_window)
                    else:
                        # fallback: switch to first handle
                        driver.switch_to.window(driver.window_handles[0])
                    print("Closed new window and switched back to:", driver.current_window_handle)
                else:
                    # If an in-page modal was used, try to click modal close button
                    try:
                        close_btns = driver.find_elements(By.CSS_SELECTOR, ".modal-dialog .close, .modal-header .close, .btn-close")
                        if close_btns:
                            driver.execute_script("arguments[0].click()", close_btns[0])
                            print("Clicked modal close button.")
                        else:
                            # fallback: press ESC
                            driver.execute_script("window.dispatchEvent(new KeyboardEvent('keydown', {'key':'Escape'}));")
                            print("Sent ESC to close modal (fallback).")
                        wait_for_loader_disappearance()
                    except Exception as e:
                        print("Could not close in-page modal cleanly:", e)
                        _dump_debug(f"modal_close_fail_q{idx+1}")
                # small pause to let page stabilise
                time.sleep(1)
            except Exception as e:
                print("Error during closing/switching back:", e)
                traceback.print_exc()
                _dump_debug(f"closing_error_q{idx+1}")

        except Exception as outer_e:
            print("Fatal error for this question:", outer_e)
            traceback.print_exc()
            _dump_debug(f"fatal_q{idx+1}")
            # Attempt to recover: close any non-original window and continue
            try:
                handles = driver.window_handles[:]
                for h in handles:
                    try:
                        if h != driver.current_window_handle:
                            driver.switch_to.window(h)
                            driver.close()
                    except Exception:
                        pass
                if len(driver.window_handles) > 0:
                    driver.switch_to.window(driver.window_handles[0])
            except Exception:
                pass
            print("Recovered, moving to next question.")
            continue

    print("DEBUG: finished subtopic")

In [11]:
def complete_course_subtopic(subtopic_number: str, subtopic_name: str, subtopic_questions: list):
    """Attempts all questions in a given subtopic with robust file fetching and window handling."""
    no_of_subtopic_questions = len(subtopic_questions)
    print(f"📚 Starting subtopic {subtopic_number}: {subtopic_name} ({no_of_subtopic_questions} questions)")
    original_window = driver.current_window_handle

    for i, question in enumerate(subtopic_questions, start=1):
        # Initialize variables
        question_file_path = None
        chosen_file = None
        answer_file = None
        in_new_window = False
        new_window_handle = None
        max_retries = 2

        try:
            # Read question info
            question_number = question.find_element(By.CLASS_NAME, "course-question-number").text
            question_name = question.find_element(By.CLASS_NAME, "course-question-name").text
            question_type = question.find_element(By.CLASS_NAME, "course-question-type").text.strip()
            question_status = bool(question.find_elements(By.CSS_SELECTOR, ".question-complete-tick-mark-mv > img"))

            print(f"\n📌 Question {i}/{no_of_subtopic_questions}: {question_number} | Type: {question_type}")

            # Skip conditions (only skip if already complete and not forced to attempt)
            if question_status and not ATTEMPT_EVEN_IF_COMPLETE:
                if question_type == "Tutorial":
                    print(f"   ⏭️  Skipping (already complete, tutorial)")
                    continue
                else:
                    print(f"   ⏭️  Skipping (already complete)")
                    continue
            
            if question_name.strip() in QUESTIONS_TO_SKIP:
                print(f"   ⏭️  Skipping (in skip list)")
                continue

            print(f"   🔄 Attempting: {question_name} (Max retries: {max_retries})")

            # Attempt with retries
            attempt = 0
            last_error = None
            
            while attempt < max_retries:
                try:
                    # Open question modal or new tab
                    driver.execute_script(
                        "arguments[0].scrollIntoView(true); arguments[0].click();",
                        question.find_element(By.CLASS_NAME, "isQuestionModal")
                    )
                    time.sleep(1)

                    # Detect new window
                    new_windows = [w for w in driver.window_handles if w != original_window]
                    in_new_window = bool(new_windows)
                    if in_new_window:
                        new_window_handle = new_windows[0]
                        driver.switch_to.window(new_window_handle)

                    # Handle tutorials
                    if question_type == "Tutorial":
                        try:
                            complete_tutorial_manual_run()
                        except Exception as e:
                            print(f"   ⚠️ Tutorial error on attempt {attempt+1}: {e}")
                            if in_new_window:
                                driver.close()
                                driver.switch_to.window(original_window)
                            attempt += 1
                            time.sleep(2)
                            continue
                        if in_new_window:
                            driver.close()
                            driver.switch_to.window(original_window)
                        print(f"   ✅ Tutorial completed")
                        break

                    # --- Resolve question file path ---
                    topic_folder = os.path.join("Testpad-Solutions", COURSE_NAME)
                    
                    # Use helper function to find subtopic folder
                    subtopic_folder_name = find_folder_by_prefix(topic_folder, f"{subtopic_number}. ")
                    if not subtopic_folder_name:
                        raise FileNotFoundError(f"No subtopic folder found for {subtopic_number}")
                    subtopic_folder = os.path.join(topic_folder, subtopic_folder_name)

                    # Use helper function to find question folder
                    question_folder_name = find_folder_by_prefix(subtopic_folder, f"{question_number}. ")
                    if not question_folder_name:
                        raise FileNotFoundError(f"No folder found for question {question_number}")
                    question_folder = os.path.join(subtopic_folder, question_folder_name)
                    
                    if attempt == 0:  # Only print once
                        print(f"   📁 Question folder: {question_folder}")
                        if os.path.isdir(question_folder):
                            print(f"   📋 Contents: {os.listdir(question_folder)}")

                    # Pick the first file with a valid extension for coding questions
                    if question_type == "Coding":
                        if attempt == 0:
                            print(f"   🔍 Looking for code file...")
                        chosen_file = find_code_file(question_folder)
                        if not chosen_file:
                            raise FileNotFoundError(f"No source file with supported extension found in {question_folder}")
                        question_file_path = chosen_file
                        if attempt == 0:
                            print(f"   ✅ Code file found: {question_file_path}")
                    else:
                        question_file_path = os.path.join(question_folder, "Answer.txt")

                    # --- Attempt answer ---
                    if question_type == "Coding":
                        print(f"   ▶️  Attempt {attempt+1}: Executing coding question...")
                        complete_coding_question(question_file_path)
                    elif question_type in ("MCQ", "Multiple Choice"):
                        print(f"   ▶️  Attempt {attempt+1}: Answering MCQ...")
                        complete_mcq_question(question_file_path)
                    elif question_type in ("MQ", "Textarea", "Multiple Questions"):
                        print(f"   ▶️  Attempt {attempt+1}: Answering textarea questions...")
                        complete_textarea_question(question_file_path)
                    else:
                        raise ValueError(f"Unknown question type: {question_type}")
                    
                    print(f"   ✅ Question completed successfully on attempt {attempt+1}")
                    break  # Success - exit retry loop

                except Exception as e:
                    last_error = str(e)
                    attempt += 1
                    print(f"   ❌ Attempt {attempt} failed: {type(e).__name__}: {e}")
                    
                    if attempt < max_retries:
                        print(f"   🔄 Retrying ({attempt}/{max_retries})...")
                        # Close window before retrying
                        if in_new_window and new_window_handle:
                            try:
                                if new_window_handle in driver.window_handles:
                                    driver.switch_to.window(new_window_handle)
                                    driver.close()
                            except:
                                pass
                        
                        # Full page refresh and recovery
                        try:
                            driver.switch_to.window(original_window)
                            driver.refresh()
                            wait_for_loader_disappearance()
                            time.sleep(2)
                        except:
                            pass
                    else:
                        print(f"   ❌ Max retries ({max_retries}) reached for this question")
                        raise
            
            # If we exit the loop without exception, question is done
            if attempt >= max_retries and last_error:
                print(f"   ❌ Failed after {max_retries} attempts: {last_error}")

        except FileNotFoundError as fnf_e:
            print(f"   ⚠️ File not found (will skip this question): {fnf_e}")
            # Skip this question only if files are missing
            if in_new_window and new_window_handle:
                try:
                    if new_window_handle in driver.window_handles:
                        driver.switch_to.window(new_window_handle)
                        driver.close()
                except:
                    pass
            continue

        except Exception as e:
            print(f"   ❌ Fatal error after retries: {type(e).__name__}: {e}")
            # Attempt recovery and continue to next
            if in_new_window and new_window_handle:
                try:
                    if new_window_handle in driver.window_handles:
                        driver.switch_to.window(new_window_handle)
                        driver.close()
                except:
                    pass
            continue

        finally:
            # Safely close new window if open and return to original
            try:
                if in_new_window and new_window_handle:
                    if new_window_handle in driver.window_handles:
                        driver.switch_to.window(new_window_handle)
                        driver.close()
                
                if original_window in driver.window_handles:
                    driver.switch_to.window(original_window)
                elif driver.window_handles:
                    driver.switch_to.window(driver.window_handles[0])
            except Exception as e:
                print(f"   ⚠️ Error during window cleanup: {e}")
            
            time.sleep(1)

In [12]:
def complete_course():
    """Iterates through all subtopics in a course and completes their questions."""
    print("📖 Starting course automation...")
    wait_for_loader_disappearance()
    time.sleep(2)

    # Fetch topics fresh each time
    course_topics = driver.find_elements(By.CLASS_NAME, "course-topics")

    print("Course subtopics:")
    for course_topic in course_topics:
        number = course_topic.find_element(By.CSS_SELECTOR, ".course-topics-number > p").text
        name = course_topic.find_element(By.CSS_SELECTOR, ".course-topics-name > p").text
        print(f"{number}) {name}")
    print("\n")

    for i in range(len(course_topics)):
        try:
            # Re-fetch topics (avoid stale element issues after each loop)
            try:
                course_topics = driver.find_elements(By.CLASS_NAME, "course-topics")
            except Exception as e:
                print(f"❌ Session lost: {e}")
                raise
            
            course_topic = course_topics[i]

            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", course_topic)

            subtopic_number = course_topic.find_element(By.CSS_SELECTOR, "div.course-topics-number > p").text
            subtopic_name = course_topic.find_element(By.CSS_SELECTOR, ".course-topics-name > p").text

            # Skip if already complete
            if not ATTEMPT_EVEN_IF_COMPLETE:
                if course_topic.find_elements(By.CSS_SELECTOR, ".icon-1"):
                    print(f"{subtopic_number}) {subtopic_name} already complete ✅")
                    continue

            # Expand subtopic if not clicked
            if "clicked" not in course_topic.get_attribute("class"):
                wait_for_loader_disappearance()
                driver.execute_script("arguments[0].click()", course_topic)

            wait_for_loader_disappearance()

            # Gather questions
            subtopic_questions = driver.find_elements(By.CLASS_NAME, "course-question")
            subtopic_questions = [q for q in subtopic_questions if q.is_displayed()]
            if not subtopic_questions:
                raise ValueError(f"{subtopic_number}) {subtopic_name} has no questions ❌")

            print("#" * 30)
            print(f"{subtopic_number}) {subtopic_name} ({len(subtopic_questions)} questions)")

            incomplete = sum(
                1 for q in subtopic_questions if not q.find_elements(By.CSS_SELECTOR, ".question-complete-tick-mark-mv > img")
            )
            print(f"{incomplete}/{len(subtopic_questions)} incomplete")

            try:
                complete_course_subtopic(subtopic_number, subtopic_name, subtopic_questions)
                print(f"{subtopic_number}) {subtopic_name} ✅ Done")
            except Exception as e:
                print(f"⚠️ Error in subtopic {subtopic_number} - {subtopic_name}: {e}")
                try:
                    driver.refresh()
                    wait_for_loader_disappearance()
                    # Retry once
                    complete_course_subtopic(subtopic_number, subtopic_name, subtopic_questions)
                except Exception as retry_e:
                    print(f"❌ Retry failed for subtopic {subtopic_number}: {retry_e}")
                    continue

            print("#" * 30, "\n")

        except Exception as subtopic_e:
            print(f"❌ Fatal error processing subtopic {i}: {subtopic_e}")
            import traceback
            traceback.print_exc()
            continue

    print(f"🎉 Course {COURSE_NAME} successfully completed ✅")

In [13]:
if __name__ == "__main__":
    try:
        print("🚀 Starting Testpad Automator...")
        login_into_testpad()
        click_course()
        complete_course()
        print("✅ Course automation finished successfully")

    except Exception as e:
        import traceback
        print("❌ Error during automation:")
        traceback.print_exc()

    finally:
        try:
            if driver:
                driver.quit()
                print("🛑 Remote browser closed")
        except Exception as e:
            print(f"⚠️ Error closing browser: {e}")


🚀 Starting Testpad Automator...
✅ Login successful
Number of subtopics - 18
📖 Starting course automation...
Course subtopics:
1) Recursion -1
2) Merge and Quick Sort
3) Bitwise Operators
4) OOPS-Introduction
5) Exception Handling
) 
) 
) 
) 
) 
) 
) 
) 
) 
) 
) 
) 
) 


1) Recursion -1 already complete ✅
2) Merge and Quick Sort already complete ✅
3) Bitwise Operators already complete ✅
4) OOPS-Introduction already complete ✅
5) Exception Handling already complete ✅
##############################
6) Inheritance-1 (27 questions)
23/27 incomplete
📚 Starting subtopic 6: Inheritance-1 (27 questions)

📌 Question 1/27: 1 | Type: Tutorial
   ⏭️  Skipping (already complete, tutorial)

📌 Question 2/27: 2 | Type: MCQ
   ⏭️  Skipping (already complete)

📌 Question 3/27: 3 | Type: MQ
   ⏭️  Skipping (already complete)

📌 Question 4/27: 4 | Type: Coding
   🔄 Attempting: Java: Inheritance - MinMaxAccount (Max retries: 2)
   📁 Question folder: Testpad-Solutions/24CSE0212-Data Structures using Object O

KeyboardInterrupt: 